# Notebook 1 — Draw the Stratified Annotation Sample

Run this **once**. It draws a 500-comment sample from the master benchmark, stratified across the
five classes and both scripts, and writes an Excel file with blank annotation columns.

**After running:** open `annotation_sample_BLANK.xlsx`, and have **two annotators independently**
fill the `annotator_1_label` and `annotator_2_label` columns (one column each, ideally by two
different people who do not see each other's answers). Use only these five labels, spelled exactly:

```
none   abusive   sexual   religious   threat
```

A one-paragraph guideline for each class is written into a second sheet (`guidelines`) of the same
file, and also printed below. Do **not** look at the hidden gold label while annotating — it is kept
in a separate file (`annotation_sample_GOLD.csv`) that Notebook 2 re-joins afterwards, so the
annotation stays blind.

No GPU needed. Runtime: a few seconds.


In [1]:
# ── Draw a 500-comment stratified sample and write a blank annotation workbook ──
import os
import numpy as np
import pandas as pd

ROOT        = os.environ.get("PROJECT_ROOT", "..")
MASTER_PATH = os.environ.get("MASTER_DATASET", f"{ROOT}/data/processed/benchmark_cleaned.csv")
OUT_DIR     = os.environ.get("KAPPA_DIR", f"{ROOT}/outputs/annotation")
os.makedirs(OUT_DIR, exist_ok=True)

N_SAMPLE   = 500
SEED       = 42
LABELS     = ["none", "abusive", "sexual", "religious", "threat"]
TEXT_COL   = "text_clean"          # falls back to "text" if absent
LABEL_COL  = "label5"

master = pd.read_csv(MASTER_PATH, encoding="utf-8-sig")
if TEXT_COL not in master.columns:
    TEXT_COL = "text"
assert {LABEL_COL, "source", "script", TEXT_COL}.issubset(master.columns), \
    f"master missing columns; found {list(master.columns)}"

# stable id so the gold labels can be re-joined blind in Notebook 2
master = master.reset_index(drop=True)
master["sample_uid"] = master.index

# Stratify jointly on (label5 × script): allocate 500 across strata proportionally,
# with a floor of a few per stratum so rare cells (e.g. threat × romanized) are represented.
rng = np.random.default_rng(SEED)
strata = master.groupby([LABEL_COL, "script"], sort=False)
sizes = strata.size()
props = sizes / sizes.sum()
alloc = (props * N_SAMPLE).round().astype(int)
alloc = alloc.clip(lower=3)                       # floor per stratum
# fix rounding so the total is exactly N_SAMPLE
while alloc.sum() != N_SAMPLE:
    diff = N_SAMPLE - alloc.sum()
    order = props.sort_values(ascending=(diff < 0)).index
    for key in order:
        if alloc.sum() == N_SAMPLE:
            break
        if diff > 0:
            alloc[key] += 1
        elif alloc[key] > 3:
            alloc[key] -= 1

picks = []
for key, grp in strata:
    k = min(int(alloc.loc[key]), len(grp))
    picks.append(grp.sample(n=k, random_state=int(rng.integers(0, 1_000_000))))
sample = pd.concat(picks).sample(frac=1.0, random_state=SEED).reset_index(drop=True)  # shuffle
sample = sample.head(N_SAMPLE)

print(f"sampled {len(sample)} comments")
print("\nby class:\n", sample[LABEL_COL].value_counts().reindex(LABELS))
print("\nby script:\n", sample["script"].value_counts())

# ---- blank annotation workbook (gold label withheld) ----
blank = pd.DataFrame({
    "sample_uid": sample["sample_uid"],
    "comment": sample[TEXT_COL].astype(str),
    "script": sample["script"],
    "annotator_1_label": "",       # <- fill manually (annotator A)
    "annotator_2_label": "",       # <- fill manually (annotator B)
})

guidelines = pd.DataFrame({
    "label": LABELS,
    "definition": [
        "No abuse, harassment, threat, or targeted hostility. Neutral, supportive, or merely off-topic.",
        "General offensive/abusive language: insults, trolling, personal attacks, slurs, body-shaming, spam-abuse, political abuse — anything hostile that is NOT specifically sexual, religious, or a threat.",
        "Sexual harassment or gender-based (misogynistic) harassment: sexual insults, objectification, gendered slurs aimed at a person.",
        "Religion-directed abuse: attacks, slurs, or hostility targeting a religion or a person because of their religion.",
        "Explicit threat or call to violence: stated intent or incitement to physically harm a person or group.",
    ],
})
# priority note for compound cases
priority = pd.DataFrame({"if a comment fits several classes, choose by this priority (highest first)":
                         ["threat", "sexual", "religious", "abusive", "none"]})

blank_path = f"{OUT_DIR}/annotation_sample_BLANK.xlsx"
with pd.ExcelWriter(blank_path, engine="openpyxl") as xl:
    blank.to_excel(xl, sheet_name="annotate", index=False)
    guidelines.to_excel(xl, sheet_name="guidelines", index=False)
    priority.to_excel(xl, sheet_name="priority_rule", index=False)

# gold labels kept separate so annotation stays blind
gold = sample[["sample_uid", LABEL_COL, TEXT_COL, "source", "script"]].rename(
    columns={LABEL_COL: "gold_label", TEXT_COL: "comment"})
gold_path = f"{OUT_DIR}/annotation_sample_GOLD.csv"
gold.to_csv(gold_path, index=False, encoding="utf-8-sig")

print(f"\n✅ wrote blank workbook -> {blank_path}")
print(f"✅ wrote hidden gold    -> {gold_path}")
print("\nGuidelines:")
for _, r in guidelines.iterrows():
    print(f"  [{r['label']}] {r['definition']}")
print("\nNext: fill annotator_1_label and annotator_2_label in the workbook, then run Notebook 2.")


sampled 500 comments

by class:
 label5
none         251
abusive      131
sexual        58
religious     44
threat        16
Name: count, dtype: int64

by script:
 script
bangla       299
romanized    201
Name: count, dtype: int64

✅ wrote blank workbook -> ../outputs/annotation/annotation_sample_BLANK.xlsx
✅ wrote hidden gold    -> ../outputs/annotation/annotation_sample_GOLD.csv

Guidelines:
  [none] No abuse, harassment, threat, or targeted hostility. Neutral, supportive, or merely off-topic.
  [abusive] General offensive/abusive language: insults, trolling, personal attacks, slurs, body-shaming, spam-abuse, political abuse — anything hostile that is NOT specifically sexual, religious, or a threat.
  [sexual] Sexual harassment or gender-based (misogynistic) harassment: sexual insults, objectification, gendered slurs aimed at a person.
  [religious] Religion-directed abuse: attacks, slurs, or hostility targeting a religion or a person because of their religion.
  [threat] Explicit th